In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt
from tqdm import tqdm
import shutil
from datetime import datetime, timedelta
import re

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [ ]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

#### Functions

In [ ]:
def str_list_to_list(str_list):
    list_out = list(map(int, re.findall(r'\d+', str_list)))
    return list_out

In [ ]:
def apply_chime_penalty(flt_ecnl, flt_factor_chime, flt_factor_nonchime, has_inst_tag):
    if has_inst_tag == 1:
        return flt_ecnl * flt_factor_chime
    else:
        return flt_ecnl * flt_factor_nonchime

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

flt_factor_24_to_72 = 2.36

int_n_payments = 4

flt_factor_chime = 1.203
flt_factor_nonchime = 0.936

str_bad_pmt_hx = 'min' # min is optimistic, max is conservative

flt_approval_rate_original = 0.20

#### Make output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [ ]:
#str_filename = 'df.gzip'
#str_uri = f's3://{str_project}/07_get_predictions/{str_filename}'
#str_uri = f's3://20250121-gen-13-model-monitoring/09_compare_account/{str_filename}'
str_uri = 's3://20250121-gen-13-model-monitoring/07_get_predictions_gen13/df_predictions.gzip'
df = pd.read_parquet(str_uri)
# set dtm
df['request_datetime'] = pd.to_datetime(df['request_datetime'])
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
# make request month
df['request_month'] = df['request_datetime'].apply(
    lambda x: str(x)[:7],
)
# rename
dict_rename = {
    'PD': 'gen13_pd',
    'LGD': 'gen13_lgd',
}
df.rename(columns=dict_rename, inplace=True)
# show
df

#### Preview

In [ ]:
list_cols = [
    'list_pmt_hx_closed__tu_pmthx',
    'list_pmt_hx_open__tu_pmthx',
]
df[list_cols]

#### Convert None to string list

In [ ]:
for col in tqdm(list_cols):
    df[col] = df[col].apply(
        lambda x: '[]' if x == 'None' else x,
    )
df[list_cols]

#### Convert to lists

In [ ]:
for col in tqdm(list_cols):
    df[col] = df[col].apply(str_list_to_list)
df[list_cols]

#### Create one long list

In [ ]:
df['list_pmt_hx'] = df.apply(
    lambda x: x['list_pmt_hx_closed__tu_pmthx'] + x['list_pmt_hx_open__tu_pmthx'],
    axis=1,
)
df[list_cols + ['list_pmt_hx']]

#### Get length of list

In [ ]:
df['n_pmts'] = df['list_pmt_hx'].apply(
    lambda x: len(x),
)
df[list_cols + ['list_pmt_hx', 'n_pmts']]

#### Tag

In [ ]:
df['tag_min_pmts'] = df['n_pmts'].apply(
    lambda x: 1 if x >= int_n_payments else 0,
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts']]

#### Get sum of last n payments

In [ ]:
df['list_last_n_pmts'] = df['list_pmt_hx'].apply(
    lambda x: x[-int_n_payments:],
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts', 'list_last_n_pmts']]

#### Get length of list n pmts

In [ ]:
df['len_last_n_pmts'] = df['list_last_n_pmts'].apply(
    lambda x: len(x),
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts', 'list_last_n_pmts', 'len_last_n_pmts']]

#### Get sum

In [ ]:
df['sum_last_n_pmts'] = df['list_last_n_pmts'].apply(
    lambda x: np.sum(x),
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts', 'list_last_n_pmts', 'len_last_n_pmts', 'sum_last_n_pmts']]

#### Tag if sum = 0 and non-bk

In [ ]:
df['tag_bad_pmt_hx'] = df.apply(
    lambda x: 1 if (x['len_last_n_pmts'] >= int_n_payments) and (x['sum_last_n_pmts'] == 0) and (x['ENG-bk'] == 0) else 0,
    axis=1,
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts', 'list_last_n_pmts', 'len_last_n_pmts', 'sum_last_n_pmts', 'ENG-bk', 'tag_bad_pmt_hx']]

#### Group by account

In [ ]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'ENG-bk': 'first',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'has_inst_tag': 'max',
    'tag_bad_pmt_hx': str_bad_pmt_hx,
})
# show
df_tmp

#### Get ECNL

In [ ]:
df_tmp['gen13_ecnl'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd'] * flt_factor_24_to_72
# show
df_tmp

#### Chime factor

In [ ]:
df_tmp['gen13_ecnl_chime'] = df_tmp.apply(
    lambda x: apply_chime_penalty(
        flt_ecnl=x['gen13_ecnl'],
        flt_factor_chime=flt_factor_chime,
        flt_factor_nonchime=flt_factor_nonchime,
        has_inst_tag=x['has_inst_tag'],
    ),
    axis=1,
)
# show
df_tmp

#### Auto decision bad pmt hx

In [ ]:
df_tmp['gen13_ecnl_chime_pmthx'] = df_tmp.apply(
    lambda x: 1.0 if (x['tag_bad_pmt_hx'] == 1) else x['gen13_ecnl_chime'],
    axis=1,
)
# show
df_tmp

#### Approved tags

In [ ]:
df_tmp['tag_approved_gen13'] = df_tmp['gen13_ecnl'].apply(
    lambda x: 1 if x <= 0.35 else 0,
)
flt_mn_original = df_tmp['tag_approved_gen13'].mean()
print(f'Proportion Approved Gen 13 at {flt_approval_rate_original*100:0.2f}% approval rate: {flt_mn_original:0.4f}')

In [ ]:
df_tmp['tag_approved_gen13_chime'] = df_tmp['gen13_ecnl_chime'].apply(
    lambda x: 1 if x <= 0.35 else 0,
)
flt_mn = df_tmp['tag_approved_gen13_chime'].mean()
flt_mn = (flt_mn * flt_approval_rate_original) / flt_mn_original
print(f'Percent Approved Gen 13 with Chime Penalty: {flt_mn*100:0.4f}%')

In [ ]:
df_tmp['tag_approved_gen13_chime_pmthx'] = df_tmp['gen13_ecnl_chime_pmthx'].apply(
    lambda x: 1 if x <= 0.35 else 0,
)
flt_mn = df_tmp['tag_approved_gen13_chime_pmthx'].mean()
flt_mn = (flt_mn * flt_approval_rate_original) / flt_mn_original
print(f'Percent Approved Gen 13 with Chime Penalty and Pmt Hx Penalty: {flt_mn*100:0.4f}%')